# mini-vLLM â€” GPU Correctness & Benchmark Notebook

**Runtime:** GPU T4 â€” Kaggle: Settings â†’ Accelerator â†’ GPU T4 x1

In [ ]:
# ── Clone / update ──────────────────────────────────────────────────────────
import os
from pathlib import Path

BRANCH    = 'milestone/m6-api-server'
CLONE_URL = 'https://github.com/shlokkvaishnav/LLM-Inference-Engine.git'
# Absolute path prevents os.chdir from nesting deeper on each re-run.
LOCAL_DIR = Path('/kaggle/working/mini-vllm')

if LOCAL_DIR.exists():
    !git -C {LOCAL_DIR} fetch origin
    !git -C {LOCAL_DIR} checkout {BRANCH}
    !git -C {LOCAL_DIR} pull
else:
    !git clone --branch {BRANCH} {CLONE_URL} {LOCAL_DIR}

os.chdir(LOCAL_DIR)
print('Working dir:', os.getcwd())

In [ ]:
# Register our package on sys.path without touching any pre-installed packages.
!pip install -e . --no-deps -q
print('Install complete.')

In [ ]:
# transformers==4.46.3 version-checks these packages at import time
# (dependency_versions_check.py). Two have upper bounds that Kaggle violates:
#   tokenizers:      needs >=0.20,<0.21  — Kaggle has 0.22.x
#   huggingface-hub: needs >=0.23.2,<1.0 — Kaggle has 1.x
# Everything else (safetensors, accelerate, tqdm, numpy, …) only has a lower
# bound so Kaggle's newer versions are fine.
#
# Strategy: install exact pins into an isolated --target dir and prepend it
# to sys.path. --upgrade avoids the "directory already exists" warning on
# re-runs. Evict sys.modules so a prior failed import cannot leave stale
# cached entries that bypass our path.
import subprocess, sys, os

TARGET = "/kaggle/working/pkgs"
os.makedirs(TARGET, exist_ok=True)

for pkg in [
    "transformers==4.46.3",
    "tokenizers==0.20.3",
    "huggingface_hub==0.26.2",
]:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install",
        pkg, "--no-deps", "-q", "--target", TARGET, "--upgrade",
    ])

# Evict stale cached modules before the next import.
_prefixes = ("transformers", "tokenizers", "huggingface_hub")
for _k in [k for k in sys.modules
           if any(k == p or k.startswith(p + ".") for p in _prefixes)]:
    del sys.modules[_k]

if TARGET not in sys.path:
    sys.path.insert(0, TARGET)

print("pinned: transformers==4.46.3  tokenizers==0.20.3  huggingface_hub==0.26.2")

In [ ]:
import torch, transformers
print(f'PyTorch      : {torch.__version__}')
print(f'transformers : {transformers.__version__}')
print(f'CUDA         : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU          : {torch.cuda.get_device_name(0)}')
    print(f'VRAM         : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
!MINI_VLLM_TEST_MODEL=TinyLlama/TinyLlama-1.1B-Chat-v1.0 MINI_VLLM_TEST_DEVICE=cuda PYTHONPATH=/kaggle/working/pkgs python -m pytest tests/test_correctness.py -v -s 2>&1

---
## M3 — Continuous batching ✓ (Milestone 3 landed)

`test_continuous_batch_matches_single`: 4 prompts through LLMEngine with max_batch_size=2.
The engine admits 2, then as each finishes it admits the next one.
Each sequence's output must match the HuggingFace greedy baseline token-for-token.

Key design points for interviews:
- `step()` runs **every decode step** (tight loop), not once per batch
- `decode_one()` is O(N) passes/step — deliberate simplicity; M4 fixes this with paged attention
- `Scheduler.free()` immediately opens a slot; next `step()` admits a waiter

## M4 — Paged KV-cache

Three parts, all on this branch:
- **BlockManager** (`mini_vllm/kv_cache/block_manager.py`) — fixed-size block pool, allocate/free/append_slot
- **Scheduler preemption** (`mini_vllm/engine/scheduler.py`) — evicts the youngest running sequence (LIFO) when it needs a block and none are free, requeues it at the front of waiting
- **Paged attention kernel** (`mini_vllm/kv_cache/paged_attention.py`) — reference (CPU) + Triton (GPU, fused, online-softmax)

`test_block_manager.py` and `test_scheduler.py` are pure Python (no GPU needed) — already verified in CI.
The cell below runs everything including the Triton GPU kernel test, which only runs here on Kaggle.

In [ ]:
# Kaggle GPU images usually ship triton already (torch.compile depends on it).
# Only install if missing, and --no-deps so it can't drag in a conflicting torch.
import importlib.util
if importlib.util.find_spec("triton") is None:
    !pip install triton --no-deps -q
import triton
print(f'triton: {triton.__version__}')

In [ ]:
!PYTHONPATH=/kaggle/working/pkgs python -m pytest tests/test_block_manager.py tests/test_scheduler.py tests/test_paged_attention.py -v -s 2>&1

### M4 engine integration — PagedLlamaRunner + batched decode

`PagedLlamaRunner` (`mini_vllm/model/paged_llama_runner.py`) wires the block pool and
Triton kernel into an actual generation path: RMSNorm/RoPE/SwiGLU/Linear layers call the
loaded HF model's own submodules directly (inherits correctness from HF), only the
attention core is replaced with the paged kernel. `LLMEngine` now calls `decode_batch()`
once per step for the WHOLE running batch — O(1) passes/step instead of M3's O(N).

Two correctness tests:
- `test_paged_llama_matches_dense_generate` — tiny random Llama (GQA config), paged vs dense, CPU-safe
- `test_paged_llama_matches_hf_on_real_model` — **real TinyLlama-1.1B on GPU**, paged path (Triton kernel)
  vs `transformers.generate()`, token-for-token. Only runs here (needs CUDA + model download).

In [ ]:
!MINI_VLLM_TEST_MODEL='TinyLlama/TinyLlama-1.1B-Chat-v1.0' \
 MINI_VLLM_TEST_DEVICE='cuda' \
 PYTHONPATH=/kaggle/working/pkgs \
 python -m pytest tests/test_paged_llama_runner.py -v -s 2>&1

---
## M5 — Quantization (INT8 / INT4)

Weight-only symmetric per-channel quantization (`mini_vllm/quantization/`) — dequantizes on
the fly before each matmul. This shrinks model size (~4x for INT8, ~8x for INT4 vs fp32) but
is **not assumed** to speed up decode — a real speedup needs a fused low-precision kernel
that never materializes the full-precision weight. The report below measures speed rather
than asserting it.

`test_quantization.py` uses bounded-error thresholds (not exact match — quantization is
lossy by construction), calibrated from measured error on realistic-magnitude weights.

In [ ]:
!PYTHONPATH=/kaggle/working/pkgs python -m pytest tests/test_quantization.py -v -s 2>&1

### Quantization report — size / quality / speed on real TinyLlama-1.1B

Three independent measurements — read them separately, don't assume one implies another.

In [ ]:
!MINI_VLLM_TEST_MODEL='TinyLlama/TinyLlama-1.1B-Chat-v1.0' \
 MINI_VLLM_TEST_DEVICE='cuda' \
 PYTHONPATH=/kaggle/working/pkgs \
 python benchmarks/quantization_report.py 2>&1

---
## M6 — FastAPI server with SSE streaming

`AsyncLLMEngine` (`mini_vllm/engine/async_engine.py`) wraps `LLMEngine` for serving: one
background asyncio task repeatedly calls `step()`, pushing each scheduled sequence's new
token into its own queue — this is continuous batching visible at the API layer, since a
request submitted mid-stream is admitted into the very next step.

`server.py` exposes `POST /v1/completions` (OpenAI-shaped, streaming or not). At startup it
picks `PagedLlamaRunner` (M4, paged attention) for Llama models on CUDA, `ModelRunner`
(M1-M3, dense) otherwise.

The CPU tests (GPT-2) already prove correctness — this cell re-runs them here with
`MINI_VLLM_MODEL=TinyLlama` + `MINI_VLLM_DEVICE=cuda`, which switches the server onto the
real production model and the `PagedLlamaRunner` + Triton kernel path, end to end.

In [ ]:
!MINI_VLLM_MODEL='TinyLlama/TinyLlama-1.1B-Chat-v1.0' \
 MINI_VLLM_DEVICE='cuda' \
 PYTHONPATH=/kaggle/working/pkgs \
 python -m pytest tests/test_api_server.py -v -s 2>&1